# Benchmark de segmentación e identificación FDI — 4 modelos × CV 5-fold por niño
**Settings:** GPU T4 x2, **Internet ON** (pesos preentrenados).
**Input:** salida de `datasets-limpios-icdas-segmentacion` (o el dataset `ahmed-caries-curated`): `fdi_segmentation_curated.zip` + `segmentation_manifest.csv`. Opcional: salida de una corrida anterior, para reanudar.

**Modelos** (una o dos por familia):
| Familia | Modelo | Preentrenamiento |
|---|---|---|
| CNN encoder-decoder | U-Net++ (ResNet50) | ImageNet-1K |
| CNN, convoluciones dilatadas | DeepLabV3+ (ResNet50) | ImageNet-1K |
| Transformer | SegFormer-B2 (MiT-B2) | ImageNet-1K |
| Instancias, tiempo real | YOLO11m-seg | COCO (el único disponible para YOLO) |

**Protocolo común:**
- 630 imágenes, 96 niños; CV 5-fold por niño con los folds del manifiesto (los mismos niños caen en el mismo fold que en ICDAS). Ronda k: prueba = fold k, validación = fold (k+1) mod 5, entrenamiento = los otros 3.
- **Sin flip vertical. Flip horizontal solo intercambiando izquierda↔derecha en las etiquetas** (11↔21, 41↔31…); en YOLO, `fliplr = 0`. Rotación ≤ 15°.
- Early stopping por **Dice medio de validación** (dientes); el fold de prueba se evalúa una vez, a la resolución nativa de la máscara.
- Métricas en prueba: Dice e IoU por diente, Dice del diente vs fondo (sin numeración), **identificación FDI** (sensibilidad, VPP, F1 por diente), **exactitud de numeración**, HD95; IC 95 % por bootstrap de niños; comparación pareada con Holm; subgrupos por vista y retractor; eficiencia.

In [ ]:
!pip -q install segmentation-models-pytorch ultralytics
import os
os.environ['PYTORCH_NVML_BASED_CUDA_CHECK'] = '1'   # no inicializar CUDA en el proceso principal (cada fold corre en un proceso hijo)
import glob, json, time, zipfile, shutil, random, math, platform, re, gc, psutil
import multiprocessing as mp
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn, torch.nn.functional as F
import albumentations as A
import segmentation_models_pytorch as smp
from transformers import SegformerForSemanticSegmentation
from scipy import ndimage
from concurrent.futures import ThreadPoolExecutor
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; NGPU = torch.cuda.device_count()
print(torch.__version__, smp.__version__, A.__version__, NGPU, 'GPU')

RUN_MODELS = ['unetpp_r50', 'deeplabv3p_r50', 'segformer_b2', 'yolo11m_seg']
RUN_FOLDS  = [0, 1, 2, 3, 4]
SEED, WORKERS, SZ, YSZ = 42, 2, 512, 640
SEM_WORKERS = 0   # augmentations en el proceso principal (las V1–V3 mostraron fuga de RAM con workers)
TIME_LIMIT_H = 11.3
IN, W, TMP = '/kaggle/input', '/kaggle/working', '/kaggle/tmp'
T0 = time.time()

## Recetas

In [ ]:
RECIPES = {
 'unetpp_r50':     dict(kind='smp', arch='UnetPlusPlus', encoder='resnet50', lr=3e-4, enc_lr_mult=0.33, wd=1e-4, bs=8, epochs=80, patience=20, warmup=3),
 'deeplabv3p_r50': dict(kind='smp', arch='DeepLabV3Plus', encoder='resnet50', lr=3e-4, enc_lr_mult=0.33, wd=1e-4, bs=8, epochs=80, patience=20, warmup=3),
 'segformer_b2':   dict(kind='hf', ckpt='nvidia/mit-b2', lr=6e-5, enc_lr_mult=1.0, head_lr_mult=10.0, wd=0.01, bs=8, epochs=80, patience=20, warmup=3),
 'yolo11m_seg':    dict(kind='yolo', ckpt='yolo11m-seg.pt', epochs=150, patience=40, bs=16, lr0=0.01, imgsz=YSZ),
}
# Loss de los semánticos: CE + Dice (multiclase). AdamW, warmup lineal + coseno, AMP. YOLO: receta por defecto de Ultralytics salvo flips.

## Datos y clases

In [ ]:
def find(name):
    h = glob.glob(f'{IN}/**/{name}', recursive=True); return h[0] if h else None
man = pd.read_csv(find('segmentation_manifest.csv'))
DATA = f'{TMP}/seg'
if not os.path.exists(f'{DATA}/fdi_segmentation_curated') and not os.path.exists(f'{DATA}/images'):
    z = find('fdi_segmentation_curated.zip')
    if z: zipfile.ZipFile(z).extractall(DATA)
    else: os.makedirs(DATA, exist_ok=True); os.symlink(os.path.dirname(os.path.dirname(find(man.image_file[0].split('/')[-1]))), f'{DATA}/x')
ROOT = os.path.dirname(os.path.dirname(glob.glob(f'{DATA}/**/{man.image_file[0]}', recursive=True)[0]))
LM = glob.glob(f'{DATA}/**/labelmap.txt', recursive=True)[0] if glob.glob(f'{DATA}/**/labelmap.txt', recursive=True) else find('labelmap.txt')
print('raíz:', ROOT, '| labelmap:', LM)
lines = [l.strip() for l in open(LM, encoding='utf-8') if l.strip() and not l.startswith('#')]
NAMES, COLORS = [], []
for l in lines:   # formato CVAT: nombre:r,g,b:partes:acciones
    p = l.split(':'); NAMES.append(p[0].strip()); COLORS.append(tuple(int(v) for v in p[1].split(',')) if len(p) > 1 and p[1] else None)
print(len(NAMES), 'clases en labelmap:', list(zip(NAMES, COLORS))[:6], '...')
bg = [i for i, n in enumerate(NAMES) if n.lower() in ('background', 'fondo', 'bg')]
if bg and bg[0] != 0:   # fondo siempre índice 0
    i = bg[0]; NAMES.insert(0, NAMES.pop(i)); COLORS.insert(0, COLORS.pop(i))
NC = len(NAMES)

m0 = cv2.imread(os.path.join(ROOT, man.mask_file[0]), cv2.IMREAD_UNCHANGED)
RGB_MASK = m0.ndim == 3
print('máscara', m0.shape, m0.dtype, '| RGB' if RGB_MASK else '| índice')
LUT = {c: i for i, c in enumerate(COLORS) if c is not None}
def read_mask(rel):
    m = cv2.imread(os.path.join(ROOT, rel), cv2.IMREAD_UNCHANGED)
    if not RGB_MASK: return m.astype(np.uint8)
    m = cv2.cvtColor(m[..., :3], cv2.COLOR_BGR2RGB); key = (m[..., 0].astype(np.int32) << 16) | (m[..., 1].astype(np.int32) << 8) | m[..., 2]
    out = np.zeros(m.shape[:2], np.uint8); seen = set(np.unique(key))
    for c, i in LUT.items():
        k = (c[0] << 16) | (c[1] << 8) | c[2]
        if k in seen: out[key == k] = i
    unk = seen - {(c[0] << 16) | (c[1] << 8) | c[2] for c in LUT}
    for u in unk: UNKNOWN[u] = UNKNOWN.get(u, 0) + int((key == u).sum())
    return out
UNKNOWN = {}
with ThreadPoolExecutor(8) as ex:
    IMGS = dict(zip(man.image_id, ex.map(lambda r: cv2.cvtColor(cv2.imread(os.path.join(ROOT, r)), cv2.COLOR_BGR2RGB), man.image_file)))
    MASKS = dict(zip(man.image_id, ex.map(read_mask, man.mask_file)))
print(len(IMGS), 'imágenes; colores desconocidos en máscaras:', len(UNKNOWN), '→ quedan como fondo')
tot_px = sum(m.size for m in MASKS.values())
print({(u >> 16, (u >> 8) & 255, u & 255): n for u, n in sorted(UNKNOWN.items(), key=lambda x: -x[1])}, f'| {sum(UNKNOWN.values()) / tot_px:.5%} de los píxeles')
cnt = np.zeros(NC, int)
for m in MASKS.values(): cnt += np.bincount(m.ravel(), minlength=NC)[:NC] > 0
print('imágenes con cada clase:', dict(zip(NAMES, cnt.tolist())))
assert man.groupby('child_id').fold.nunique().max() == 1
print(pd.crosstab(man.fold, man.view, margins=True))

### Numeración e intercambio izquierda↔derecha
El `labelmap` usa numeración **Universal** (1–32). Se convierte a FDI para reportar. El flip horizontal intercambia: Universal 1–16 ↔ 17−n (arcada superior) y 17–32 ↔ 49−n (inferior).

In [ ]:
def univ(n):
    m = re.search(r'\d+', n); return int(m.group()) if m else None
def to_fdi(u):
    if u is None or not 1 <= u <= 32: return None
    if u <= 8: return 10 + (9 - u)
    if u <= 16: return 20 + (u - 8)
    if u <= 24: return 30 + (25 - u)
    return 40 + (u - 24)
U = [univ(n) for n in NAMES]; FDI = [to_fdi(u) if i else None for i, u in enumerate(U)]
print('Universal → FDI:', {NAMES[i]: FDI[i] for i in range(1, NC)})
assert all(f is not None for f in FDI[1:]), 'nombres del labelmap no numéricos: revisar antes de usar flip'
SWAP = np.arange(NC, dtype=np.uint8)
u2i = {u: i for i, u in enumerate(U) if i}
for i in range(1, NC):
    u = U[i]; v = 17 - u if u <= 16 else 49 - u
    SWAP[i] = u2i.get(v, i)
print('pares intercambiados (FDI):', sorted({tuple(sorted((FDI[i], FDI[SWAP[i]]))) for i in range(1, NC) if SWAP[i] != i})[:8], '...')
TEETH = list(range(1, NC))

## Augmentations y datasets (U-Net++, DeepLabV3+, SegFormer)

In [ ]:
MEAN, STD = np.array([0.485, 0.456, 0.406], np.float32), np.array([0.229, 0.224, 0.225], np.float32)
AUG = A.Compose([A.RandomResizedCrop(size=(SZ, SZ), scale=(0.6, 1.0), ratio=(0.8, 1.25)),
                 A.Affine(rotate=(-15, 15), scale=(0.9, 1.1), translate_percent=(-0.05, 0.05), border_mode=cv2.BORDER_CONSTANT, p=0.5),
                 A.RandomBrightnessContrast(0.2, 0.2, p=0.5), A.HueSaturationValue(8, 15, 10, p=0.4),
                 A.OneOf([A.GaussianBlur(blur_limit=(3, 5)), A.GaussNoise(std_range=(0.02, 0.06))], p=0.25)])
class SegDS(torch.utils.data.Dataset):
    def __init__(self, ids, train): self.ids, self.train = list(ids), train
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        im, m = IMGS[self.ids[i]], MASKS[self.ids[i]]
        if self.train:
            if random.random() < 0.5: im, m = im[:, ::-1].copy(), SWAP[m[:, ::-1]]   # flip con intercambio izq↔der
            o = AUG(image=im, mask=m); im, m = o['image'], o['mask']
        else:
            im = cv2.resize(im, (SZ, SZ), interpolation=cv2.INTER_LINEAR); m = cv2.resize(m, (SZ, SZ), interpolation=cv2.INTER_NEAREST)
        x = torch.from_numpy(((im.astype(np.float32) / 255 - MEAN) / STD).transpose(2, 0, 1))
        return x, torch.from_numpy(m.astype(np.int64))

## Modelos, loss y entrenamiento

In [ ]:
class HFSeg(nn.Module):
    def __init__(self, ckpt):
        super().__init__(); self.m = SegformerForSemanticSegmentation.from_pretrained(ckpt, num_labels=NC, ignore_mismatched_sizes=True)
    def forward(self, x): return F.interpolate(self.m(pixel_values=x).logits, size=x.shape[-2:], mode='bilinear', align_corners=False)
def build(r, pretrained=True):
    if r['kind'] == 'smp':
        return getattr(smp, r['arch'])(encoder_name=r['encoder'], encoder_weights='imagenet' if pretrained else None, classes=NC)
    return HFSeg(r['ckpt'])
def param_groups(net, r):
    enc, head = [], []
    for n, p in net.named_parameters():
        (enc if ('encoder' in n and 'decode' not in n) else head).append(p)
    return [dict(params=enc, lr=r['lr'] * r['enc_lr_mult']), dict(params=head, lr=r['lr'] * r.get('head_lr_mult', 1.0))]
def dice_loss(logits, y):
    p = logits.float().softmax(1); oh = F.one_hot(y, NC).permute(0, 3, 1, 2).float()
    inter = (p * oh).sum((0, 2, 3)); den = p.sum((0, 2, 3)) + oh.sum((0, 2, 3)); present = oh.sum((0, 2, 3)) > 0
    d = 1 - (2 * inter + 1) / (den + 1); return d[1:][present[1:]].mean() if present[1:].any() else d[1:].mean()
def lr_mult(r, e):
    if e < r['warmup']: return (e + 1) / r['warmup']
    return 0.5 * (1 + math.cos(math.pi * (e - r['warmup']) / max(1, r['epochs'] - r['warmup'])))
def seed_all(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def predict_native(net, ids, bs=8):
    """Predice a 512 y lleva los logits a la resolución nativa de cada máscara."""
    net.eval(); out = {}
    for i in range(0, len(ids), bs):
        b = ids[i:i + bs]
        x = torch.stack([torch.from_numpy(((cv2.resize(IMGS[j], (SZ, SZ)).astype(np.float32) / 255 - MEAN) / STD).transpose(2, 0, 1)) for j in b]).to(DEV)
        with torch.autocast('cuda', enabled=DEV == 'cuda'): lg = net(x).float()
        for j, l in zip(b, lg):
            h, w = MASKS[j].shape; out[j] = F.interpolate(l[None], size=(h, w), mode='bilinear', align_corners=False)[0].argmax(0).byte().cpu().numpy()
    return out
def mean_dice(pred, ids):
    I = np.zeros(NC); S = np.zeros(NC)
    for j in ids:
        g, p = MASKS[j], pred[j]
        I += np.bincount(g[g == p], minlength=NC)[:NC]; S += np.bincount(g.ravel(), minlength=NC)[:NC] + np.bincount(p.ravel(), minlength=NC)[:NC]
    d = 2 * I / np.maximum(S, 1); return d[1:][S[1:] > 0].mean()

def preds_path(mname, k): return f'{W}/results/{mname}/fold{k}_preds.npz'
def save_preds(mname, k, pred):   # un archivo por fold en el output (sobrevive a una corrida que falla)
    np.savez_compressed(preds_path(mname, k), **{str(j): p for j, p in pred.items()})
import ctypes
try: LIBC = ctypes.CDLL('libc.so.6')
except OSError: LIBC = None
def free_ram():
    gc.collect()
    if LIBC is not None: LIBC.malloc_trim(0)   # devuelve al sistema la memoria liberada (fragmentación de glibc)
def ram():
    me = psutil.Process(); rss = me.memory_info().rss + sum(c.memory_info().rss for c in me.children(recursive=True))
    shm = shutil.disk_usage('/dev/shm').used if os.path.exists('/dev/shm') else 0
    return f'RAM {psutil.virtual_memory().used / 2**30:.1f}/{psutil.virtual_memory().total / 2**30:.0f} GB (proceso {rss / 2**30:.1f}, shm {shm / 2**30:.1f})'
RAM_EPOCH_STOP_GB = 27.0   # red de seguridad: si se acerca al límite, se cierra el fold con el mejor checkpoint (queda marcado en meta)

def run_sem(mname, k):
    out = f'{W}/results/{mname}'; os.makedirs(out, exist_ok=True)
    r = RECIPES[mname]
    if os.path.exists(f'{out}/fold{k}_meta.json'):
        if not os.path.exists(preds_path(mname, k)) and os.path.exists(f'{out}/fold{k}_weights_fp16.pt'):   # regenerar predicciones desde los pesos
            net = build(r, pretrained=False); net.load_state_dict({n: v.float() if v.is_floating_point() else v for n, v in torch.load(f'{out}/fold{k}_weights_fp16.pt').items()})
            net.to(DEV); te = man[man.fold == k]; save_preds(mname, k, predict_native(net, te.image_id.tolist(), bs=4))
            print('predicciones regeneradas', mname, k); del net; torch.cuda.empty_cache()
        else: print('ya existe', mname, k)
        return
    seed_all(SEED)
    te, va = man[man.fold == k], man[man.fold == (k + 1) % 5]; tr = man[~man.fold.isin([k, (k + 1) % 5])]
    dl = torch.utils.data.DataLoader(SegDS(tr.image_id, True), batch_size=r['bs'], shuffle=True, num_workers=SEM_WORKERS, drop_last=True,
                                     pin_memory=False, persistent_workers=SEM_WORKERS > 0, worker_init_fn=lambda i: (np.random.seed(SEED + i), random.seed(SEED + i)))
    net = build(r).to(DEV); base = net
    # sin DataParallel: con 2 GPU y cuDNN, DeepLabV3+ iba 4,5× más lento y perdía ~1,4 GB de RAM por época (diagnóstico 22-oct);
    # en su lugar, dos folds corren en paralelo, uno por GPU
    opt = torch.optim.AdamW(param_groups(base, r), weight_decay=r['wd']); lrs0 = [g['lr'] for g in opt.param_groups]
    scaler = torch.amp.GradScaler(enabled=DEV == 'cuda'); ce = nn.CrossEntropyLoss()
    if DEV == 'cuda': torch.cuda.reset_peak_memory_stats()
    best, best_ep, bad, hist, t_train, stopped_ram = -1, 0, 0, [], time.time(), False
    for ep in range(r['epochs']):
        for g, l0 in zip(opt.param_groups, lrs0): g['lr'] = l0 * lr_mult(r, ep)
        net.train(); t = time.time(); L = 0
        for x, y in dl:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            with torch.autocast('cuda', enabled=DEV == 'cuda'):
                lg = net(x); loss = ce(lg.float(), y) + dice_loss(lg, y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); scaler.step(opt); scaler.update(); L += loss.item()
        vd = mean_dice(predict_native(net, va.image_id.tolist()), va.image_id)
        free_ram(); hist.append(dict(epoch=ep + 1, loss=L / len(dl), val_dice=vd, sec=time.time() - t, ram_gb=psutil.virtual_memory().used / 2**30))
        print(f'{mname} f{k} ep{ep + 1:02d} loss {L / len(dl):.3f} valDice {vd:.3f} ({time.time() - t:.0f}s)' + (f' | {ram()}' if ep % 5 == 0 or ep < 3 else ''))
        if vd > best: best, best_ep, bad = vd, ep + 1, 0; torch.save(base.state_dict(), f'{TMP}/best_{mname}_{k}.pt')
        else:
            bad += 1
            if bad >= r['patience']: break
        if (time.time() - T0) / 3600 > TIME_LIMIT_H: print('límite de tiempo: se descarta esta ronda'); return
        if psutil.virtual_memory().used / 2**30 > RAM_EPOCH_STOP_GB: print('RAM al límite: se cierra el fold con el mejor checkpoint', ram()); stopped_ram = True; break
    t_train = time.time() - t_train
    base.load_state_dict(torch.load(f'{TMP}/best_{mname}_{k}.pt')); t = time.time()
    pred = predict_native(base, te.image_id.tolist(), bs=4); ms = 1000 * (time.time() - t) / len(te)
    save_preds(mname, k, pred)
    torch.save({n: v.half() if v.is_floating_point() else v for n, v in base.state_dict().items()}, f'{out}/fold{k}_weights_fp16.pt')
    pd.DataFrame(hist).to_csv(f'{out}/fold{k}_history.csv', index=False)
    json.dump(dict(model=mname, fold=k, best_val_dice=best, best_epoch=best_ep, epochs_run=len(hist), stopped_by_ram=stopped_ram, train_time_s=t_train, gpu_ms_per_img=ms,
                   params=sum(p.numel() for p in base.parameters()), vram_peak_mb=torch.cuda.max_memory_allocated() / 2**20 if DEV == 'cuda' else None,
                   recipe=r, names=NAMES, fdi=FDI, env=dict(torch=torch.__version__, smp=smp.__version__, python=platform.python_version())),
              open(f'{out}/fold{k}_meta.json', 'w'), indent=1)
    print('>>', mname, 'fold', k, '| best ep', best_ep, '| test Dice', round(mean_dice(pred, te.image_id), 3))
    del net, base, dl, opt; free_ram(); torch.cuda.empty_cache(); print(ram())

## YOLO11m-seg
Máscaras → polígonos por componente conexo (una instancia por diente). Entrenamiento a 640 sin flips. Para evaluar, las instancias se pintan en orden de confianza creciente sobre el fondo y se llevan a la resolución nativa: así se evalúa con las mismas métricas que los semánticos.

In [ ]:
def yolo_labels():
    d = f'{TMP}/yolo'
    if os.path.exists(f'{d}/done'): return d
    os.makedirs(f'{d}/images', exist_ok=True); os.makedirs(f'{d}/labels', exist_ok=True)
    for j in man.image_id:
        im, m = IMGS[j], MASKS[j]; h, w = m.shape
        cv2.imwrite(f'{d}/images/{j}.jpg', cv2.cvtColor(cv2.resize(im, (w, h)), cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])
        rows = []
        for c in np.unique(m):
            if c == 0: continue
            n, lab = cv2.connectedComponents((m == c).astype(np.uint8))
            for q in range(1, n):
                comp = (lab == q).astype(np.uint8)
                if comp.sum() < 30: continue
                cs, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE); cs = max(cs, key=cv2.contourArea).reshape(-1, 2)
                if len(cs) < 3: continue
                rows.append(f'{c - 1} ' + ' '.join(f'{x / w:.5f} {y / h:.5f}' for x, y in cs))
        open(f'{d}/labels/{j}.txt', 'w').write('\n'.join(rows))
    open(f'{d}/done', 'w').write('ok'); return d

def yolo_predict(best, ids, r, d):
    pred = {}
    for j in ids:
        h, w = MASKS[j].shape; res = best.predict(f'{d}/images/{j}.jpg', imgsz=r['imgsz'], conf=0.25, retina_masks=True, verbose=False, device=0)[0]
        p = np.zeros((h, w), np.uint8)
        if res.masks is not None:
            ms = res.masks.data.cpu().numpy(); cl = res.boxes.cls.cpu().numpy().astype(int); cf = res.boxes.conf.cpu().numpy()
            for q in np.argsort(cf):
                mm = cv2.resize(ms[q], (w, h), interpolation=cv2.INTER_NEAREST) > 0.5; p[mm] = cl[q] + 1
        pred[j] = p
    return pred

from ultralytics import YOLO
def run_yolo(mname, k):
    out = f'{W}/results/{mname}'; os.makedirs(out, exist_ok=True)
    r = RECIPES[mname]; d = yolo_labels()
    if os.path.exists(f'{out}/fold{k}_meta.json'):
        if not os.path.exists(preds_path(mname, k)) and os.path.exists(f'{out}/fold{k}_best.pt'):
            save_preds(mname, k, yolo_predict(YOLO(f'{out}/fold{k}_best.pt'), man[man.fold == k].image_id, r, d)); print('predicciones regeneradas', mname, k)
        else: print('ya existe', mname, k)
        return
    te, va = man[man.fold == k], man[man.fold == (k + 1) % 5]; tr = man[~man.fold.isin([k, (k + 1) % 5])]
    for s, df in (('train', tr), ('val', va)): open(f'{d}/{s}_{k}.txt', 'w').write('\n'.join(f'{d}/images/{j}.jpg' for j in df.image_id))
    yml = f'{d}/fold{k}.yaml'
    open(yml, 'w').write(f'path: {d}\ntrain: {d}/train_{k}.txt\nval: {d}/val_{k}.txt\nnames:\n' + ''.join(f'  {i - 1}: "{FDI[i]}"\n' for i in TEETH))
    hrs_left = TIME_LIMIT_H - (time.time() - T0) / 3600
    if hrs_left < 1.0: print('sin tiempo para', mname, k); return
    seed_all(SEED); t_train = time.time()
    model = YOLO(r['ckpt'])
    model.train(data=yml, epochs=r['epochs'], patience=r['patience'], imgsz=r['imgsz'], batch=r['bs'], lr0=r['lr0'], device=0, workers=WORKERS,
                fliplr=0.0, flipud=0.0, degrees=10.0, seed=SEED, deterministic=False, project=f'{TMP}/yolo_runs', name=f'f{k}', exist_ok=True,
                plots=False, verbose=False)
    t_train = time.time() - t_train
    best = YOLO(f'{TMP}/yolo_runs/f{k}/weights/best.pt'); t = time.time(); pred = yolo_predict(best, te.image_id, r, d)
    ms_img = 1000 * (time.time() - t) / len(te); save_preds(mname, k, pred)
    shutil.copy2(f'{TMP}/yolo_runs/f{k}/weights/best.pt', f'{out}/fold{k}_best.pt')
    hist = pd.read_csv(f'{TMP}/yolo_runs/f{k}/results.csv'); hist.to_csv(f'{out}/fold{k}_history.csv', index=False)
    json.dump(dict(model=mname, fold=k, epochs_run=len(hist), train_time_s=t_train, gpu_ms_per_img=ms_img, recipe=r, names=NAMES, fdi=FDI),
              open(f'{out}/fold{k}_meta.json', 'w'), indent=1)
    print('>>', mname, 'fold', k, '| test Dice', round(mean_dice(pred, te.image_id), 3))

## Reanudar y correr

In [ ]:
for p in glob.glob(f'{IN}/**/results/*/fold*_meta.json', recursive=True):
    m = p.split('/')[-2]; os.makedirs(f'{W}/results/{m}', exist_ok=True)
    for f in glob.glob(os.path.dirname(p) + '/*'):
        if not os.path.exists(f'{W}/results/{m}/' + os.path.basename(f)): shutil.copy2(f, f'{W}/results/{m}/')
print('ya hechos:', sorted(glob.glob(f'{W}/results/*/fold*_meta.json')))
RAM_STOP_GB = 22.0   # no se lanzan folds nuevos si la RAM ya está alta
def child(fn, mname, k, gpu):
    os.environ['CUDA_VISIBLE_DEVICES'] = str(gpu)   # cada proceso ve una sola GPU
    fn(mname, k)
if any(RECIPES[m]['kind'] == 'yolo' for m in RUN_MODELS): yolo_labels()   # se prepara una vez, antes de los procesos en paralelo
jobs = [(m, k) for m in RUN_MODELS for k in RUN_FOLDS]
slots = list(range(max(1, NGPU))); running = {}
while jobs or running:
    for g, (pr, m, k) in list(running.items()):
        if not pr.is_alive():
            pr.join(); print(f'[{m} fold {k} | GPU {g}] terminó con código {pr.exitcode} | {ram()}')
            if pr.exitcode != 0 and not os.path.exists(f'{W}/results/{m}/fold{k}_meta.json'): print('  el fold falló; revisar el log')
            del running[g]
    free = [g for g in slots if g not in running]
    can_start = (time.time() - T0) / 3600 < TIME_LIMIT_H - 0.5 and psutil.virtual_memory().used / 2**30 < RAM_STOP_GB
    while jobs and free and can_start:
        m, k = jobs.pop(0)
        if os.path.exists(f'{W}/results/{m}/fold{k}_meta.json') and os.path.exists(preds_path(m, k)): print('ya existe', m, k); continue
        fn = run_yolo if RECIPES[m]['kind'] == 'yolo' else run_sem
        g = free.pop(0); pr = mp.get_context('fork').Process(target=child, args=(fn, m, k, g)); pr.start(); running[g] = (pr, m, k)
        print(f'[{m} fold {k}] empieza en GPU {g}')
    if jobs and not can_start and not running: print('se detiene: límite de tiempo o RAM; relanzar con esta salida como input'); break
    time.sleep(10)
print('entrenamiento', round((time.time() - T0) / 3600, 2), 'h')

## Eficiencia (una vez por modelo): parámetros, GFLOPs y latencia CPU (1 imagen, 1 hilo)

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
eff = []
for m in RECIPES:
    f = f'{W}/results/{m}/efficiency.json'
    if not os.path.exists(f'{W}/results/{m}'): continue
    if os.path.exists(f): eff.append(json.load(open(f))); continue
    r = RECIPES[m]
    if r['kind'] == 'yolo':
        from ultralytics import YOLO
        pt = sorted(glob.glob(f'{W}/results/{m}/fold*_best.pt'))
        if not pt: continue
        net = YOLO(pt[0]).model.float().eval(); x = torch.randn(1, 3, YSZ, YSZ)
    else:
        net = build(r).eval(); x = torch.randn(1, 3, SZ, SZ)
    with FlopCounterMode(display=False) as fc, torch.no_grad(): net(x)
    torch.set_num_threads(1)
    with torch.no_grad():
        for _ in range(2): net(x)
        t = time.time()
        for _ in range(10): net(x)
    torch.set_num_threads(os.cpu_count())
    e = dict(model=m, params_M=sum(p.numel() for p in net.parameters()) / 1e6, gflops=fc.get_total_flops() / 2e9,
             cpu_ms_per_img_1thread=1000 * (time.time() - t) / 10, input=x.shape[-1])
    json.dump(e, open(f, 'w')); eff.append(e)
print(pd.DataFrame(eff).round(2).to_string())

## Métricas en prueba (resolución nativa)
Por imagen y diente (clase presente en la referencia o en la predicción):
- **Dice / IoU** por diente; **Dice diente-vs-fondo** (ignora el número).
- **Identificación FDI:** un diente de la referencia está *identificado* si el Dice de su clase ≥ 0,5. Sensibilidad = identificados / dientes de la referencia; VPP = clases predichas (≥ 50 px) con Dice ≥ 0,5 / clases predichas; F1.
- **Numeración:** entre los dientes de la referencia cubiertos ≥ 50 % por alguna predicción de diente, fracción cuya clase predicha mayoritaria es la correcta.
- **HD95** (px) para dientes presentes en ambos.
- Agregados: Dice e IoU por clase (sumando sobre imágenes) y su media sobre dientes; IC 95 % por bootstrap de niños (1.000); comparación pareada contra el mejor (Holm); subgrupos por vista y por retractor.

In [ ]:
def hd95(a, b):
    if not a.any() or not b.any(): return np.nan
    ea, eb = a ^ ndimage.binary_erosion(a), b ^ ndimage.binary_erosion(b)
    da, db = ndimage.distance_transform_edt(~eb), ndimage.distance_transform_edt(~ea)
    return np.percentile(np.concatenate([da[ea], db[eb]]), 95)
def img_stats(j, p):
    g = MASKS[j]; rows = []
    fg = dict(fg_dice=2 * ((g > 0) & (p > 0)).sum() / max(1, (g > 0).sum() + (p > 0).sum()))
    for c in TEETH:
        G, P = g == c, p == c; ng, npx = G.sum(), P.sum()
        if ng == 0 and npx < 50: continue
        inter = (G & P).sum(); d = 2 * inter / max(1, ng + npx)
        maj = np.bincount(p[G], minlength=NC) if ng else None
        covered = ng > 0 and (p[G] > 0).mean() >= 0.5
        rows.append(dict(image_id=j, tooth=FDI[c], cls=c, in_gt=ng > 0, in_pred=npx >= 50, inter=inter, gt_px=ng, pred_px=npx, dice=d,
                         iou=inter / max(1, ng + npx - inter), identified=(ng > 0) and d >= 0.5, tp_pred=(npx >= 50) and d >= 0.5,
                         covered=covered, numbered_ok=covered and maj[1:].argmax() + 1 == c,
                         hd95=hd95(G, P) if ng and npx else np.nan, **fg))
    return rows
def agg(T):
    g, pr = T[T.in_gt], T[T.in_pred]
    per_cls = T.groupby('cls').apply(lambda d: pd.Series(dict(I=d.inter.sum(), S=d.gt_px.sum() + d.pred_px.sum())), include_groups=False)
    per_cls = per_cls[per_cls.S > 0]; dc = 2 * per_cls.I / per_cls.S; ic = per_cls.I / (per_cls.S - per_cls.I)
    sens = g.identified.mean(); ppv = pr.tp_pred.mean() if len(pr) else np.nan
    return dict(dice_tooth=dc.mean(), iou_tooth=ic.mean(), dice_img=g.groupby('image_id').dice.mean().mean(),
                fg_dice=T.groupby('image_id').fg_dice.first().mean(), id_sens=sens, id_ppv=ppv, id_f1=(2 * sens * ppv / (sens + ppv)) if (sens + ppv) > 0 else 0.0,
                numbering_acc=g[g.covered].numbered_ok.mean(), hd95=T.hd95.median())
rows = []
for m in RECIPES:
    for k in range(5):
        f = preds_path(m, k)
        if not os.path.exists(f): continue
        P = np.load(f)
        for j in man[man.fold == k].image_id:
            if str(j) in P: rows += [dict(model=m, fold=k, **r) for r in img_stats(j, P[str(j)])]
T = pd.DataFrame(rows)
if len(T):
    T = T.merge(man[['image_id', 'child_id', 'view', 'condition']], on='image_id'); T.to_csv(f'{W}/results/dientes_por_imagen.csv', index=False)
    per = pd.DataFrame([dict(model=m, fold=k, **agg(d)) for (m, k), d in T.groupby(['model', 'fold'])]); per.to_csv(f'{W}/results/metricas_por_fold.csv', index=False)
    print('Por fold (media ± DE):'); print(per.groupby('model').agg(['mean', 'std']).drop(columns='fold').round(3).to_string())
    done = [m for m, d in T.groupby('model') if d.fold.nunique() == 5]; print('Modelos con 5 folds:', done)
    rng = np.random.default_rng(0); kids = np.array(sorted(man.child_id.unique())); B = 1000
    draws = [rng.choice(kids, len(kids)) for _ in range(B)]; res, boots = [], {}
    for m in done:
        g = {c: d for c, d in T[T.model == m].groupby('child_id')}
        bs = pd.DataFrame([agg(pd.concat([g[c] for c in dr if c in g])) for dr in draws]); boots[m] = bs; pt = agg(T[T.model == m])
        res.append({'model': m, **pt, **{f'{c}_lo': bs[c].quantile(.025) for c in bs}, **{f'{c}_hi': bs[c].quantile(.975) for c in bs}})
    if res:
        res = pd.DataFrame(res).sort_values('dice_tooth', ascending=False); res.to_csv(f'{W}/results/metricas_ic95.csv', index=False)
        print(res[['model'] + [f'{c}{s}' for c in ['dice_tooth', 'fg_dice', 'id_f1', 'numbering_acc', 'hd95'] for s in ('', '_lo', '_hi')]].round(3).to_string())
        if len(done) > 1:
            top = res.model.iloc[0]; cmp = []
            for m in done:
                if m == top: continue
                dl_ = boots[top].dice_tooth.values - boots[m].dice_tooth.values
                cmp.append(dict(vs=m, diff=dl_.mean(), lo=np.quantile(dl_, .025), hi=np.quantile(dl_, .975), p=min(1, 2 * min((dl_ <= 0).mean(), (dl_ >= 0).mean()))))
            cmp = pd.DataFrame(cmp).sort_values('p'); n = len(cmp)
            cmp['p_holm'] = np.minimum(1, np.maximum.accumulate([(n - i) * p for i, p in enumerate(cmp.p)]))
            print(f'Dice por diente de {top} vs cada modelo (bootstrap pareado por niño):'); print(cmp.round(3).to_string()); cmp.to_csv(f'{W}/results/comparacion_pareada.csv', index=False)
    sub = pd.DataFrame([dict(model=m, grupo=f'{c}={v}', n_img=d.image_id.nunique(), **agg(d)) for c in ('view', 'condition') for (m, v), d in T.groupby(['model', c])])
    sub.to_csv(f'{W}/results/metricas_subgrupos.csv', index=False); print(sub[['model', 'grupo', 'n_img', 'dice_tooth', 'id_f1', 'numbering_acc']].round(3).to_string())
    pt = T[T.in_gt].groupby(['model', 'tooth']).agg(n=('dice', 'size'), dice=('dice', 'mean'), identified=('identified', 'mean')).reset_index()
    pt.to_csv(f'{W}/results/metricas_por_diente.csv', index=False)
print('tiempo total', round((time.time() - T0) / 3600, 2), 'h')